In [1]:
import pandas as pd
import numpy as np
import os
import gc

In [2]:
RAW_PATH = "../data/raw/"
PROCESSED_PATH = "../data/processed/"

os.makedirs(PROCESSED_PATH, exist_ok=True)

tracking_path = RAW_PATH + "PlayerTrackData.csv"
playlist_path = RAW_PATH + "PlayList.csv"
injury_path = PROCESSED_PATH + "injury_play_metadata.csv"

print("Paths configured.")

Paths configured.


In [3]:
playlist = pd.read_csv(
    playlist_path,
    dtype={"PlayKey": "string"}
)

injury_metadata = pd.read_csv(
    injury_path,
    dtype={"PlayKey": "string"}
)

print("Playlist shape:", playlist.shape)
print("Injury metadata shape:", injury_metadata.shape)

display(playlist.head())
display(injury_metadata.head())

Playlist shape: (267005, 14)
Injury metadata shape: (77, 23)


,PlayerKey,GameID,PlayKey,RosterPosition,PlayerDay,PlayerGame,StadiumType,FieldType,Temperature,Weather,PlayType,PlayerGamePlay,Position,PositionGroup
0,26624,26624-1,26624-1-1,Quarterback,1,1,Outdoor,Synthetic,63,Clear and warm,Pass,1,QB,QB
1,26624,26624-1,26624-1-2,Quarterback,1,1,Outdoor,Synthetic,63,Clear and warm,Pass,2,QB,QB
2,26624,26624-1,26624-1-3,Quarterback,1,1,Outdoor,Synthetic,63,Clear and warm,Rush,3,QB,QB
3,26624,26624-1,26624-1-4,Quarterback,1,1,Outdoor,Synthetic,63,Clear and warm,Rush,4,QB,QB
4,26624,26624-1,26624-1-5,Quarterback,1,1,Outdoor,Synthetic,63,Clear and warm,Pass,5,QB,QB


,PlayerKey_injury,GameID_injury,PlayKey,BodyPart,Surface,DM_M1,DM_M7,DM_M28,DM_M42,PlayerKey_playlist,...,PlayerGame,StadiumType,FieldType,Temperature,Weather,PlayType,PlayerGamePlay,Position,PositionGroup,_merge
0,39873,39873-4,39873-4-32,Knee,Synthetic,1,1,1,1,39873,...,4,Indoors,Synthetic,84,Cloudy,Punt,32,OLB,LB,both
1,46074,46074-7,46074-7-26,Knee,Natural,1,1,0,0,46074,...,7,Open,Natural,76,Partly Cloudy,Punt,26,OLB,LB,both
2,36557,36557-1,36557-1-70,Ankle,Synthetic,1,1,1,1,36557,...,1,Outdoor,Synthetic,63,Clear and warm,Pass,70,SS,DB,both
3,46646,46646-3,46646-3-30,Ankle,Natural,1,0,0,0,46646,...,3,Outdoor,Natural,80,Cloudy,Punt,30,LB,LB,both
4,43532,43532-5,43532-5-69,Ankle,Synthetic,1,1,1,1,43532,...,5,Retractable Roof,Synthetic,89,Partly Cloudy,Kickoff,69,WR,WR,both


In [4]:
tracking_sample = pd.read_csv(
    tracking_path,
    nrows=5
)

print("Tracking columns:")
print(tracking_sample.columns.tolist())

display(tracking_sample)

Tracking columns:
['PlayKey', 'time', 'event', 'x', 'y', 'dir', 'dis', 'o', 's']


,PlayKey,time,event,x,y,dir,dis,o,s
0,26624-1-1,0.0,huddle_start_offense,87.46,28.93,288.24,0.01,262.33,0.13
1,26624-1-1,0.1,NaN,87.45,28.92,283.91,0.01,261.69,0.12
2,26624-1-1,0.2,NaN,87.44,28.92,280.40,0.01,261.17,0.12
3,26624-1-1,0.3,NaN,87.44,28.92,278.79,0.01,260.66,0.10
4,26624-1-1,0.4,NaN,87.44,28.92,275.44,0.01,260.27,0.09


In [5]:
tracking_features_list = []

chunksize = 1_000_000

for chunk_number, chunk in enumerate(
    pd.read_csv(
        tracking_path,
        usecols=[
            "PlayKey", "time", "x", "y",
            "dis", "s", "dir", "o"
        ],
        dtype={"PlayKey": "string"},
        chunksize=chunksize
    ),
    start=1
):

    print(f"Processing chunk {chunk_number}")

    # Remove rows without PlayKey
    chunk = chunk.dropna(subset=["PlayKey"])

    # Convert numerical columns
    numeric_cols = [
        "time", "x", "y", "dis",
        "s", "dir", "o"
    ]

    for col in numeric_cols:
        chunk[col] = pd.to_numeric(
            chunk[col], errors="coerce"
        )

    # Sort observations within each PlayKey
    chunk = chunk.sort_values(
        ["PlayKey", "time"]
    )

    # Calculate time difference
    chunk["time_diff"] = (
        chunk.groupby("PlayKey")["time"].diff()
    )

    # Calculate speed difference
    chunk["speed_diff"] = (
        chunk.groupby("PlayKey")["s"].diff()
    )

    # Calculate acceleration
    chunk["acceleration"] = (
        chunk["speed_diff"] / chunk["time_diff"]
    )

    # Remove infinite acceleration values
    chunk["acceleration"] = (
        chunk["acceleration"].replace(
            [np.inf, -np.inf], np.nan
        )
    )

    chunk["acceleration_abs"] = (
        chunk["acceleration"].abs()
    )

    # Create partial features for each PlayKey
    grouped = chunk.groupby("PlayKey").agg(
        tracking_points=("time", "count"),

        start_time=("time", "min"),
        end_time=("time", "max"),

        distance_total=("dis", "sum"),

        speed_sum=("s", "sum"),
        speed_sq_sum=("s", lambda x: (x.dropna() ** 2).sum()),
        speed_count=("s", "count"),
        speed_max=("s", "max"),
        speed_min=("s", "min"),

        acceleration_sum=("acceleration", "sum"),
        acceleration_abs_sum=("acceleration_abs", "sum"),
        acceleration_count=("acceleration", "count"),
        acceleration_abs_max=("acceleration_abs", "max"),

        x_min=("x", "min"),
        x_max=("x", "max"),
        y_min=("y", "min"),
        y_max=("y", "max")
    ).reset_index()

    tracking_features_list.append(grouped)

    del chunk, grouped
    gc.collect()

print("Tracking feature extraction completed.")

Processing chunk 1
Processing chunk 2
Processing chunk 3
Processing chunk 4
Processing chunk 5
Processing chunk 6
Processing chunk 7
Processing chunk 8
Processing chunk 9
Processing chunk 10
Processing chunk 11
Processing chunk 12
Processing chunk 13
Processing chunk 14
Processing chunk 15
Processing chunk 16
Processing chunk 17
Processing chunk 18
Processing chunk 19
Processing chunk 20
Processing chunk 21
Processing chunk 22
Processing chunk 23
Processing chunk 24
Processing chunk 25
Processing chunk 26
Processing chunk 27
Processing chunk 28
Processing chunk 29
Processing chunk 30
Processing chunk 31
Processing chunk 32
Processing chunk 33
Processing chunk 34
Processing chunk 35
Processing chunk 36
Processing chunk 37
Processing chunk 38
Processing chunk 39
Processing chunk 40
Processing chunk 41
Processing chunk 42
Processing chunk 43
Processing chunk 44
Processing chunk 45
Processing chunk 46
Processing chunk 47
Processing chunk 48
Processing chunk 49
Processing chunk 50
Processin

In [6]:
partial_features = pd.concat(
    tracking_features_list,
    ignore_index=True
)

print("Partial feature shape:", partial_features.shape)

tracking_features = (
    partial_features.groupby("PlayKey")
    .agg(
        tracking_points=("tracking_points", "sum"),

        start_time=("start_time", "min"),
        end_time=("end_time", "max"),

        distance_total=("distance_total", "sum"),

        speed_sum=("speed_sum", "sum"),
        speed_sq_sum=("speed_sq_sum", "sum"),
        speed_count=("speed_count", "sum"),
        speed_max=("speed_max", "max"),
        speed_min=("speed_min", "min"),

        acceleration_sum=("acceleration_sum", "sum"),
        acceleration_abs_sum=("acceleration_abs_sum", "sum"),
        acceleration_count=("acceleration_count", "sum"),
        acceleration_abs_max=("acceleration_abs_max", "max"),

        x_min=("x_min", "min"),
        x_max=("x_max", "max"),
        y_min=("y_min", "min"),
        y_max=("y_max", "max")
    )
    .reset_index()
)

print("Final tracking features:", tracking_features.shape)
display(tracking_features.head())

Partial feature shape: (267036, 18)
Final tracking features: (266960, 18)


,PlayKey,tracking_points,start_time,end_time,distance_total,speed_sum,speed_sq_sum,speed_count,speed_max,speed_min,acceleration_sum,acceleration_abs_sum,acceleration_count,acceleration_abs_max,x_min,x_max,y_min,y_max
0,26624-1-1,299,0.0,29.8,16.83,160.19,196.3237,299,2.94,0.00,-0.1,101.1,298,2.7,84.36,91.49,23.62,29.85
1,26624-1-10,276,0.0,27.5,17.07,159.06,205.3256,276,3.18,0.00,-4.8,111.4,275,2.5,13.39,18.56,22.07,27.49
2,26624-1-11,369,0.0,36.8,31.20,293.64,434.2036,369,2.94,0.01,23.9,150.1,368,2.1,7.16,19.65,22.56,28.18
3,26624-1-12,357,0.0,35.6,14.70,124.84,108.9478,357,1.83,0.00,1.8,88.2,356,2.6,45.55,49.93,20.87,24.66
4,26624-1-13,256,0.0,25.5,13.26,109.34,88.4892,256,1.49,0.01,1.4,86.8,255,2.8,45.20,49.21,21.53,23.97


In [7]:
# Play duration
tracking_features["duration"] = (
    tracking_features["end_time"] -
    tracking_features["start_time"]
)

# Average speed
tracking_features["speed_mean"] = (
    tracking_features["speed_sum"] /
    tracking_features["speed_count"]
)

# Speed standard deviation
variance = (
    tracking_features["speed_sq_sum"] /
    tracking_features["speed_count"]
    - tracking_features["speed_mean"] ** 2
)

tracking_features["speed_std"] = np.sqrt(
    variance.clip(lower=0)
)

# Average acceleration
tracking_features["acceleration_mean"] = (
    tracking_features["acceleration_sum"] /
    tracking_features["acceleration_count"]
)

# Average absolute acceleration
tracking_features["acceleration_abs_mean"] = (
    tracking_features["acceleration_abs_sum"] /
    tracking_features["acceleration_count"]
)

# Position ranges
tracking_features["x_range"] = (
    tracking_features["x_max"] -
    tracking_features["x_min"]
)

tracking_features["y_range"] = (
    tracking_features["y_max"] -
    tracking_features["y_min"]
)

# Remove temporary calculation columns
tracking_features.drop(
    columns=[
        "speed_sum", "speed_sq_sum", "speed_count",
        "acceleration_sum", "acceleration_abs_sum",
        "acceleration_count"
    ],
    inplace=True
)

display(tracking_features.head())

,PlayKey,tracking_points,start_time,end_time,distance_total,speed_max,speed_min,acceleration_abs_max,x_min,x_max,y_min,y_max,duration,speed_mean,speed_std,acceleration_mean,acceleration_abs_mean,x_range,y_range
0,26624-1-1,299,0.0,29.8,16.83,2.94,0.00,2.7,84.36,91.49,23.62,29.85,29.8,0.535753,0.607923,-0.000336,0.339262,7.13,6.23
1,26624-1-10,276,0.0,27.5,17.07,3.18,0.00,2.5,13.39,18.56,22.07,27.49,27.5,0.576304,0.641722,-0.017455,0.405091,5.17,5.42
2,26624-1-11,369,0.0,36.8,31.20,2.94,0.01,2.1,7.16,19.65,22.56,28.18,36.8,0.795772,0.737191,0.064946,0.407880,12.49,5.62
3,26624-1-12,357,0.0,35.6,14.70,1.83,0.00,2.6,45.55,49.93,20.87,24.66,35.6,0.349692,0.427658,0.005056,0.247753,4.38,3.79
4,26624-1-13,256,0.0,25.5,13.26,1.49,0.01,2.8,45.20,49.21,21.53,23.97,25.5,0.427109,0.404028,0.005490,0.340392,4.01,2.44


In [8]:
print("Feature dataset shape:", tracking_features.shape)

print("\nMissing values:")
display(tracking_features.isnull().sum())

print("\nInfinite values:")
print(
    np.isinf(
        tracking_features.select_dtypes(
            include=np.number
        )
    ).sum()
)

Feature dataset shape: (266960, 19)

Missing values:


PlayKey                  0
tracking_points          0
start_time               0
end_time                 0
distance_total           0
speed_max                0
speed_min                0
acceleration_abs_max     0
x_min                    0
x_max                    0
y_min                    0
y_max                    0
duration                 0
speed_mean               0
speed_std                0
acceleration_mean        0
acceleration_abs_mean    0
x_range                  0
y_range                  0
dtype: int64


Infinite values:
tracking_points          0
start_time               0
end_time                 0
distance_total           0
speed_max                0
speed_min                0
acceleration_abs_max     0
x_min                    0
x_max                    0
y_min                    0
y_max                    0
duration                 0
speed_mean               0
speed_std                0
acceleration_mean        0
acceleration_abs_mean    0
x_range                  0
y_range                  0
dtype: int64


In [9]:
tracking_features.to_csv(
    PROCESSED_PATH + "tracking_play_features.csv",
    index=False
)

print("Tracking features saved successfully.")

Tracking features saved successfully.


In [10]:
assert tracking_features["PlayKey"].is_unique
assert playlist["PlayKey"].is_unique

play_features = playlist.merge(
    tracking_features,
    on="PlayKey",
    how="left",
    validate="one_to_one"
)

print("Play-level feature shape:", play_features.shape)

display(play_features.head())

Play-level feature shape: (267005, 32)


,PlayerKey,GameID,PlayKey,RosterPosition,PlayerDay,PlayerGame,StadiumType,FieldType,Temperature,Weather,...,x_max,y_min,y_max,duration,speed_mean,speed_std,acceleration_mean,acceleration_abs_mean,x_range,y_range
0,26624,26624-1,26624-1-1,Quarterback,1,1,Outdoor,Synthetic,63,Clear and warm,...,91.49,23.62,29.85,29.8,0.535753,0.607923,-0.000336,0.339262,7.13,6.23
1,26624,26624-1,26624-1-2,Quarterback,1,1,Outdoor,Synthetic,63,Clear and warm,...,90.73,27.35,29.96,25.9,0.849692,0.617141,0.041313,0.386486,10.18,2.61
2,26624,26624-1,26624-1-3,Quarterback,1,1,Outdoor,Synthetic,63,Clear and warm,...,81.67,25.74,29.63,29.5,0.331014,0.487729,0.043729,0.158983,4.33,3.89
3,26624,26624-1,26624-1-4,Quarterback,1,1,Outdoor,Synthetic,63,Clear and warm,...,74.56,24.63,27.71,12.6,0.523701,0.529241,0.075397,0.353175,3.09,3.08
4,26624,26624-1,26624-1-5,Quarterback,1,1,Outdoor,Synthetic,63,Clear and warm,...,71.19,21.93,26.41,25.9,0.970385,0.869971,0.049807,0.268340,20.47,4.48


In [11]:
# Standardize FieldType
play_features["FieldType"] = (
    play_features["FieldType"]
    .astype("string")
    .str.strip()
    .str.lower()
)

# Encode surface
play_features["is_synthetic"] = (
    play_features["FieldType"].map({
        "synthetic": 1,
        "natural": 0
    })
)

# Convert temperature to numeric
play_features["Temperature"] = pd.to_numeric(
    play_features["Temperature"],
    errors="coerce"
)

# Create temperature categories
play_features["temperature_category"] = pd.cut(
    play_features["Temperature"],
    bins=[-np.inf, 32, 50, 70, 85, np.inf],
    labels=[
        "Very Cold",
        "Cold",
        "Moderate",
        "Warm",
        "Hot"
    ]
)

# Standardize position group
play_features["PositionGroup"] = (
    play_features["PositionGroup"]
    .astype("string")
    .str.strip()
    .str.upper()
)

display(
    play_features[
        [
            "PlayKey",
            "FieldType",
            "is_synthetic",
            "Temperature",
            "temperature_category",
            "PositionGroup"
        ]
    ].head()
)

,PlayKey,FieldType,is_synthetic,Temperature,temperature_category,PositionGroup
0,26624-1-1,synthetic,1,63,Moderate,QB
1,26624-1-2,synthetic,1,63,Moderate,QB
2,26624-1-3,synthetic,1,63,Moderate,QB
3,26624-1-4,synthetic,1,63,Moderate,QB
4,26624-1-5,synthetic,1,63,Moderate,QB


In [12]:
injury_enriched = injury_metadata.merge(
    tracking_features,
    on="PlayKey",
    how="left",
    validate="many_to_one"
)

print("Injury enriched shape:", injury_enriched.shape)

display(injury_enriched.head())

Injury enriched shape: (77, 41)


,PlayerKey_injury,GameID_injury,PlayKey,BodyPart,Surface,DM_M1,DM_M7,DM_M28,DM_M42,PlayerKey_playlist,...,x_max,y_min,y_max,duration,speed_mean,speed_std,acceleration_mean,acceleration_abs_mean,x_range,y_range
0,39873,39873-4,39873-4-32,Knee,Synthetic,1,1,1,1,39873,...,93.45,17.30,46.63,44.1,1.489140,2.417293,-0.016780,0.580952,37.27,29.33
1,46074,46074-7,46074-7-26,Knee,Natural,1,1,0,0,46074,...,95.29,23.66,28.91,22.2,2.024215,2.422019,0.003604,0.836937,40.67,5.25
2,36557,36557-1,36557-1-70,Ankle,Synthetic,1,1,1,1,36557,...,84.01,10.64,30.59,15.9,1.574938,1.981710,-0.020126,0.810063,10.71,19.95
3,46646,46646-3,46646-3-30,Ankle,Natural,1,0,0,0,46646,...,71.11,18.55,31.13,24.9,2.194320,2.605092,0.001205,0.698394,47.09,12.58
4,43532,43532-5,43532-5-69,Ankle,Synthetic,1,1,1,1,43532,...,75.47,45.27,49.23,34.9,0.760543,1.204567,-0.006017,0.408883,19.96,3.96


In [13]:
play_features.to_csv(
    PROCESSED_PATH + "play_level_features.csv",
    index=False
)

injury_enriched.to_csv(
    PROCESSED_PATH + "injury_enriched_features.csv",
    index=False
)

print("All feature engineering datasets saved.")

All feature engineering datasets saved.


In [15]:
print("Play-level dataset:")
print(play_features.shape)

print("\nInjury-enriched dataset:")
print(injury_enriched.shape)

print("\nSurface distribution:")
print(play_features["FieldType"].value_counts(dropna=False))

print("\nTracking feature availability:")
print(
    play_features["speed_mean"].notna().sum(),
    "plays have speed information"
)

print("\nMissing tracking features:")
display(
    play_features[
        [
            "tracking_points",
            "distance_total",
            "speed_mean",
            "speed_max",
            "acceleration_abs_mean"
        ]
    ].isnull().sum()
)

Play-level dataset:
(267005, 34)

Injury-enriched dataset:
(77, 41)

Surface distribution:
FieldType
natural      156902
synthetic    110103
Name: count, dtype: int64[pyarrow]

Tracking feature availability:
266960 plays have speed information

Missing tracking features:


tracking_points          45
distance_total           45
speed_mean               45
speed_max                45
acceleration_abs_mean    45
dtype: int64